# Search Engine With Tools and AI Agents

Building a LangChain agent with Wikipedia, ArXiv, and custom LangSmith documentation search.

In [2]:
import os

# Set your Hugging Face token if needed for embeddings
os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")

# Groq API key will be loaded later

In [5]:
# ────────────────────────────────────────────────
#              Import required packages
# ────────────────────────────────────────────────

from langchain_community.tools import WikipediaQueryRun, ArxivQueryRun
from langchain_community.utilities import WikipediaAPIWrapper, ArxivAPIWrapper

from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_core.tools import StructuredTool
from langchain_classic.tools.retriever import create_retriever_tool

from langchain_groq import ChatGroq
from langchain_classic import hub

from langchain_classic.agents import create_tool_calling_agent, AgentExecutor

from dotenv import load_dotenv
load_dotenv()

True

In [6]:
# ────────────────────────────────────────────────
#              Wikipedia Tool
# ────────────────────────────────────────────────

wiki_api_wrapper = WikipediaAPIWrapper(
    top_k_results=1,
    doc_content_chars_max=250
)

wiki_tool = StructuredTool.from_function(
    func=wiki_api_wrapper.run,
    name="wikipedia",
    description="A wrapper around Wikipedia. Useful for general knowledge questions about people, places, events, companies, history, etc. Input should be a clear search query.",
    return_direct=False
)

In [7]:
# ────────────────────────────────────────────────
#              ArXiv Tool
# ────────────────────────────────────────────────

arxiv_api_wrapper = ArxivAPIWrapper(
    top_k_results=1,
    doc_content_chars_max=250
)

arxiv_tool = StructuredTool.from_function(
    func=arxiv_api_wrapper.run,
    name="arxiv",
    description="A wrapper around Arxiv.org. Useful for questions about academic papers, research, preprints, authors in science and technology. Input should be a search query.",
    return_direct=False
)

In [8]:
# ────────────────────────────────────────────────
#     Custom retriever tool (LangSmith docs)
# ────────────────────────────────────────────────

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

loader = WebBaseLoader("https://docs.smith.langchain.com/")
docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
documents = text_splitter.split_documents(docs)

vectorstore = FAISS.from_documents(documents, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

retriever_tool = create_retriever_tool(
    retriever,
    "langsmith_search",
    "Searches and returns excerpts from the LangSmith documentation. Use this for questions about LangSmith, tracing, evaluation, datasets, etc."
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 218.14it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [9]:
tools = [wiki_tool, arxiv_tool, retriever_tool]
print("Available tools:", [t.name for t in tools])

Available tools: ['wikipedia', 'arxiv', 'langsmith_search']


In [10]:
# ────────────────────────────────────────────────
#                   LLM Setup
# ────────────────────────────────────────────────

groq_api_key = os.getenv("GROQ_API_KEY")

llm = ChatGroq(
    groq_api_key=groq_api_key,
    model_name="llama-3.1-8b-instant",
    temperature=0.7
)

In [12]:
# ────────────────────────────────────────────────
#               Create the Agent
# ────────────────────────────────────────────────

# Pull a good ReAct-style prompt that supports tool calling
prompt = hub.pull("hwchase17/openai-functions-agent")

agent = create_tool_calling_agent(
    llm=llm,
    tools=tools,
    prompt=prompt
)


In [14]:
## Agent Executor
from langchain_classic.agents import AgentExecutor
agent_executor = AgentExecutor(agent=agent,tools=tools,verbose=True)
agent_executor


AgentExecutor(verbose=True, agent=RunnableMultiActionAgent(runnable=RunnableAssign(mapper={
  agent_scratchpad: RunnableLambda(lambda x: message_formatter(x['intermediate_steps']))
})
| ChatPromptTemplate(input_variables=['agent_scratchpad', 'input'], optional_variables=['chat_history'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMe

In [15]:
agent_executor.invoke({"input":"Tell me about langsmith"})



> Entering new AgentExecutor chain...

Invoking: `langsmith_search` with `{'query': 'langsmith'}`


LangSmith docs - Docs by LangChainSkip to main contentDocs by LangChain home pageLangSmithSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangSmith docsGet startedObservabilityEvaluationPrompt engineeringAgent deploymentPlatform setupReferenceOverviewCreate an account and API keyIntegrationsPlansAccount administrationOverviewWorkspace setupUsers & access controlBilling & usageManage organizations using the APIToolsPolly AI assistantBetaCLISkillsAdditional resourcesData & complianceFAQLangSmith statusLangSmith docsCopy pageCopy pageLangSmith is a framework-agnostic platform for developing, debugging, and deploying AI agents and LLM applications.
It helps you trace requests, evaluate outputs, test prompts, and manage deployments in one place. LangSmith works with any agent stack, whether you’re using an existing framework or building your own. Prototype locally, then 

{'input': 'Tell me about langsmith',
 'output': 'I see that the response contained a lot more than just the answer to your question. It seems like the function call returned the entire LangSmith documentation. I will make sure to be more precise in the future.\n\nHere\'s a revised answer:\n\nThe LangSmith documentation can be found here: <function=langsmith_search>{"query": "langsmith docs"}</function>'}

In [16]:
agent_executor.invoke({"input":"What is machine learning?"})



> Entering new AgentExecutor chain...

Invoking: `wikipedia` with `{'query': 'machine learning'}`


Page: Machine learning
Summary: Machine learning (ML) is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from data and generalize to unseen data, and thus perform tasks wit
Invoking: `wikipedia` with `{'query': 'machine learning'}`
responded: It seems that the Wikipedia summary is cut off. Let me try again to get the full summary.



Page: Machine learning
Summary: Machine learning (ML) is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from data and generalize to unseen data, and thus perform tasks wit
Invoking: `wikipedia` with `{'query': 'machine learning algorithms'}`


Page: Machine learning
Summary: Machine learning (ML) is a field of study in artificial intelligence concerned with the development and study of statistical algo

{'input': 'What is machine learning?',
 'output': 'The information provided by the Wikipedia function calls gives a comprehensive overview of machine learning, including its definition, history, timeline, algorithms, applications, examples, and more.'}

In [17]:
agent_executor.invoke({"input":"What's the paper 1706.03762 about?"})



> Entering new AgentExecutor chain...

Invoking: `arxiv` with `{'query': '1706.03762'}`


Published: 2023-08-02
Title: Attention Is All You Need
Authors: Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Lukasz Kaiser, Illia Polosukhin
Summary: The dominant sequence transduction models are based on c
Invoking: `arxiv` with `{'query': '1706.03762'}`


Published: 2023-08-02
Title: Attention Is All You Need
Authors: Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Lukasz Kaiser, Illia Polosukhin
Summary: The dominant sequence transduction models are based on c
Invoking: `arxiv` with `{'query': '1706.03762 (Attention Is All You Need)'}`




HTTPError: Page request resulted in HTTP 429 (https://export.arxiv.org/api/query?search_query=1706.03762+%28Attention+Is+All+You+Need%29&id_list=&sortBy=relevance&sortOrder=descending&start=0&max_results=100)